# Setup
## Importowanie Bibliotek

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.preprocessing import RobustScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from pathlib import Path

## Pomocne Funkcje

In [ ]:
def barchart_record_count_by_category_value(dataframe, category_column_name):
    # Set visual style
    sns.set_theme(style="whitegrid")
    plt.figure(figsize=(6, 5))

    # Create count plot for the 'Survived' column
    ax = sns.countplot(
        data=titanic_df,
        x=f"{category_column_name}",
        hue=f"{category_column_name}",
        palette={0: "#e74c3c", 1: "#2ecc71"},
        legend=False,
    )

    # Customize labels and title
    plt.title(
        f"Record Count by Category: {category_column_name}", fontsize=14, fontweight="bold", pad=15
    )
    plt.xlabel("Category Values", fontsize=12)
    plt.ylabel("Record Count", fontsize=12)


    # Add numeric annotations inside bars
    for p in ax.patches:
        height = int(p.get_height())
        if height > 0:
            ax.annotate(
                f"{height}",
                (p.get_x() + p.get_width() / 2.0, height / 2),
                ha="center",
                va="center",
                fontsize=11,
                color="white",
                fontweight="bold",
            )

    plt.tight_layout()
    plt.show()

def get_scoring_metrics(expected_values, predicted_values):
    accuracy = accuracy_score(expected_values, predicted_values)
    f1score = f1_score(expected_values, predicted_values)
    _, _, _, false_positives = confusion_matrix(expected_values, predicted_values).ravel()
    return accuracy, f1score, false_positives



## Ładowanie Danych

In [ ]:
environments = ['Local', 'Colab']
environment = 'Local'

if environment not in environments:
    print("Environment Invalid!")

elif environment == 'Local':
    base_path = Path("02_podstawy_aiml") # change to path folder with data

elif environment == 'Colab':
    from google.colab import drive # remove the cell if not using colab
    drive.mount('/content/drive')
    base_path = Path("./drive/MyDrive/Solvro/")

titanic_df = pd.read_csv( base_path / 'titanic_analyzed.csv')
print(titanic_df.info())

Za wczasu usunę jeszcze kolumnę `PassengerId`:

In [ ]:
titanic_df = titanic_df.drop('PassengerId', axis='columns')
print(titanic_df.info())

## Skrót Obróbki Danych
To zrobiłem z danymi:
* `Name` i `Ticket` dropnąłem bez żadnych ekstrakcji
*  Z `Cabin` dokonałem ekstrakcji pierwszej litery w sekwencji i zastąpiłem kolumną `Location`
*  `Parch` i `SibSp` zsumowałem i zastąpiłem kolumną `Family`
* `Sex` zredukowałem do biarnej flagi `IsMale`
* `Embarked`, `Location` oraz `Pclass` encodowałem metodą one-hot
* `Fare` - zważając na outliery - transformowałem logarytmicznie w celach unormalnienia dystrybucji

## Wyniki Analizy

*Czego nauczyło Cię o badanym zbiorze danych poprzednie zadanie? Jak możesz wykorzystać wyciągnięte z niego wnioski w procesie tworzenia modelu?*

To udało mi się wydedukować:
- jeśli jesteś memszczyzną to według modelu masz o wiele mniejsze szansy przeżycia
- im wyższa klasa pasażerska (im niższa wartość `Pclass`) tym większa szansa na przeżycie
- im droższy miałeś bilet, tym bardziej prawdopodone że przeżyłeś
- najbardziej sprzyjająca lokalizacja do przetrwania katastrofy to 'B'
# Tworzenie Modelu
## Uwaga: Niezbalansowane dane

In [ ]:
barchart_record_count_by_category_value(titanic_df, 'Survived')

Będzie to potrzebne podczas rozpatrywania modeli *DummyClassifier*
## Jaki Rodzaj ML użyć?
Nasze dane mają etykiety w postaci `Survived`, zatem nasz model należeć będzie do dziedziny **uczenia nadzorowanego**.
Ponadto, wymagamy wartości nie ciągłej, a dyskretnej (przydzielić record do danej klasy), zatem będzie to zadanie **klasyfikacji**.
Zatem mamy do wyboru modele:
* Regresja Logistyczna
* Drzewa / Lasy Decyzyjne
* Support Vector Machines
* MLP
* K-Nearest Neigbhour
* XGBoost
( i jeszcze DummyClassifier)

Rozważę następujące modele:
* *Regresja Logistyczna*
* `???`

## Jakie Metryki?
* `accurary` - klasyk
* `F1-score` - kompensuje niezbalansowanie datasetu, a z takim właśnie pracujemy
* `False-Positive` - jeśli model ma mi przewidzieć, czy nie skończę jak Leonardo daVinci, to chciałbym wiedzieć jakie mam szanse że się pomylił

## Preparacja Datasetu do Treningu
### Separacja Etykiet od Predykatorów
Rolę etykiety (**target**) pełni kolumna `Survived`, więc rozdzielmy ją teraz od naszych predykatorów (**data**):

In [ ]:
dataframe_data = titanic_df.drop(columns=['Survived']) # X
dataframe_target = titanic_df['Survived'] # y

### Podział Training|Testing

In [ ]:
seed = 420
data_training, data_testing, target_training, target_testing = train_test_split(dataframe_data, dataframe_target, train_size=.8, random_state=seed ,shuffle=True)

### Baseline: DummyClassifier

In [ ]:
model = DummyClassifier(strategy="most_frequent")
model.fit(data_training, target_training)
target_prediction = model.predict(data_testing)
accuracy, f1score, false_positives = get_scoring_metrics(target_testing, target_prediction)
print(accuracy)
print(f1score)
print(false_positives)

Podczas gdy `accuracy` wprowadza jakąś użyteczną informacje, to niezbalansowanie datasetu + strategia mody
prowadzi nas do wyzerowania `F1-score` oraz `false_positives`.
`F1-score` jest trochę bardziej skomplikowany do wytłumaczenia,
natomiast `false_positives` == 0, gdyż dataset posiada więcej osób które nie przeżyły, zatem DummyClassifier
predykuje że nikt nie przeżyje - brak false-positiwów!.
Przez to że `Survived` ma więcej **0** niż **1**, to F1-score DummClassifiera musi być 0.

## Regresja Liniowa

### Standaryzacja Danych
Wybieram Robust Scaler, gdyż brzmi on jako najbardziej uniwersalny i bezbłędny (przede wszystkim odporny na outliery)

In [ ]:
scaler = RobustScaler()
data_training_scaled = scaler.fit_transform(data_training)
data_testing_scaled = scaler.transform(data_testing)

### Trenowanie Modelu

In [ ]:
model = LogisticRegression(random_state=seed)
model.fit(data_training_scaled, target_training)

### Predykcja Testowa

In [ ]:
target_prediction = model.predict(data_testing_scaled)
accuracy, f1score, false_positives = get_scoring_metrics(target_testing, target_prediction)
print(accuracy)
print(f1score)
print(false_positives)

Jak widać, dla danego zbioru testowego model osiągnął celność *81%*.

### Cross-Validacja
Upewnijmy się że wynik ten nie jest wynikiem wylosowania łatwego zbioru testowego:

In [ ]:
model = LogisticRegression(random_state=seed)
results = cross_val_score(make_pipeline(RobustScaler(),model), dataframe_data, dataframe_target, cv=10, scoring='accuracy')
print(f'Accuracy: {results}')
results = cross_val_score(make_pipeline(RobustScaler(),model), dataframe_data, dataframe_target, cv=10, scoring='f1')
print(f'F1-Score: {results}')

Cross-Validacja potwierdziła, że wytrenowany model jest dość stabilny( wartości oscylują w małej odległości od centrum).

### Wnioski
Patrząc jedynie przez przymat `accuracy`, model regresji logistycznej jest lepszy od małpiego wybierania mody zbioru danych o 23.5pkt%.
Trudno na razie wypowiedzieć się na temat `F1-score` oraz `false-positives`.